# 02 - Data Cleaning Pipeline

## Objective
Prepare the incident event log into analysis-ready datasets while preserving data meaning and traceability.

This notebook focuses on data preparation only:
- Missing value handling with documented rules
- Data type conversion and categorical standardization
- Duplicate strategy for two analysis levels (event-level and incident-level)
- Data validation and feature engineering
- Export of cleaned datasets for EDA, SQL, and dashboarding

**Outputs:**
- `data/processed/incident_event_log_clean.csv` (event-level)
- `data/processed/incident_clean.csv` (latest incident-level)

## 1. Load Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RAW_DATA = Path('../data/raw/incident_event_log.csv')
PROCESSED_DIR = Path('../data/processed')
DATE_COLUMNS = [
    'opened_at',
    'sys_created_at',
    'sys_updated_at',
    'resolved_at',
    'closed_at',
]
BOOLEAN_COLUMNS = ['active', 'made_sla', 'knowledge', 'u_priority_confirmation']

## 2. Load Dataset

In [ ]:
df_raw = pd.read_csv(
    RAW_DATA,
    na_values=['?'],
    dtype={'vendor': 'string'},
    low_memory=False,
)
df = df_raw.copy()

dataset_overview = {
    'rows': int(df_raw.shape[0]),
    'columns': int(df_raw.shape[1]),
}

display(dataset_overview)
df_raw.info()

{'rows': 141712, 'columns': 36}

<class 'pandas.DataFrame'>
RangeIndex: 141712 entries, 0 to 141711
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   number                   141712 non-null  str  
 1   incident_state           141712 non-null  str  
 2   active                   141712 non-null  bool 
 3   reassignment_count       141712 non-null  int64
 4   reopen_count             141712 non-null  int64
 5   sys_mod_count            141712 non-null  int64
 6   made_sla                 141712 non-null  bool 
 7   caller_id                141712 non-null  str  
 8   opened_by                141712 non-null  str  
 9   opened_at                141712 non-null  str  
 10  sys_created_by           141712 non-null  str  
 11  sys_created_at           141712 non-null  str  
 12  sys_updated_by           141712 non-null  str  
 13  sys_updated_at           141712 non-null  str  
 14  contact_type             141712 non-null  str  

## 3. Initial Data Quality Assessment

Assess missingness and data completeness before applying transformations.

Notes:
- `?` in the source file is interpreted as missing (`NaN`).
- Missing values can represent business reality, not always data errors.

In [3]:
df = df.replace('?', np.nan)

missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_analysis = pd.DataFrame({
    'column': missing_count.index,
    'missing_count': missing_count.values,
    'missing_pct': missing_pct.values,
    'non_null': len(df) - missing_count.values,
})

missing_analysis = (
    missing_analysis[missing_analysis['missing_pct'] > 0]
    .sort_values('missing_pct', ascending=False)
    .reset_index(drop=True)
)

total_missing = int(df.isnull().sum().sum())
total_cells = int(len(df) * len(df.columns))
missing_summary = {
    'total_missing_values': total_missing,
    'missing_rate_pct': round((total_missing / total_cells) * 100, 1),
}

display(missing_analysis)
display(missing_summary)

,column,missing_count,missing_pct,non_null
0,caused_by,141689,99.983770,23
1,vendor,141468,99.827820,244
2,cmdb_ci,141267,99.685983,445
3,rfc,140721,99.300694,991
4,problem_id,139417,98.380518,2295
5,sys_created_by,53076,37.453427,88636
6,sys_created_at,53076,37.453427,88636
7,u_symptom,32964,23.261262,108748
8,assigned_to,27496,19.402732,114216
9,assignment_group,14213,10.029496,127499


{'total_missing_values': 894597, 'missing_rate_pct': 17.5}

## 4. Missing Values

In [4]:
na_preserve_fields = [
    'vendor', 'problem_id', 'rfc', 'caused_by', 'cmdb_ci',
    'resolved_at', 'closed_at', 'assignment_group', 'assigned_to',
    'category', 'subcategory', 'u_symptom',
]

user_fields = ['opened_by', 'sys_created_by', 'sys_updated_by']
for col in user_fields:
    df[col] = df[col].fillna('Unassigned')

categorical_fill = {'location': 'Unknown', 'closed_code': 'Unknown', 'caller_id': 'Unknown'}
for col, value in categorical_fill.items():
    df[col] = df[col].fillna(value)

missing_decisions = pd.DataFrame({
    'rule': ['preserve_nan', 'fill_unassigned', 'fill_unknown'],
    'columns_count': [len(na_preserve_fields), len(user_fields), len(categorical_fill)],
})

missing_decisions

,rule,columns_count
0,preserve_nan,12
1,fill_unassigned,3
2,fill_unknown,3


## 5. Data Type Conversion

In [5]:
for col in DATE_COLUMNS:
    df[col] = pd.to_datetime(
        df[col],
        format='%d/%m/%Y %H:%M',
        errors='coerce',
    )

df[DATE_COLUMNS].dtypes

opened_at         datetime64[us]
sys_created_at    datetime64[us]
sys_updated_at    datetime64[us]
resolved_at       datetime64[us]
closed_at         datetime64[us]
dtype: object

## 6. Categorical Standardization

In [6]:
df['incident_state'] = df['incident_state'].replace('-100', 'Unknown')

def extract_priority_level(value):
    if pd.isna(value):
        return np.nan
    try:
        return int(str(value).split(' - ')[0])
    except (ValueError, IndexError):
        return np.nan

def extract_priority_text(value):
    if pd.isna(value):
        return np.nan
    try:
        return str(value).split(' - ')[1]
    except (ValueError, IndexError):
        return np.nan

df['impact_level'] = df['impact'].apply(extract_priority_level)
df['impact_text'] = df['impact'].apply(extract_priority_text)
df['urgency_level'] = df['urgency'].apply(extract_priority_level)
df['urgency_text'] = df['urgency'].apply(extract_priority_text)
df['priority_level'] = df['priority'].apply(extract_priority_level)
df['priority_text'] = df['priority'].apply(extract_priority_text)

for col in BOOLEAN_COLUMNS:
    df[col] = df[col].astype(bool)

{
    'incident_state_values': sorted(df['incident_state'].dropna().unique().tolist()),
    'boolean_columns': BOOLEAN_COLUMNS,
}

{'incident_state_values': ['Active',
  'Awaiting Evidence',
  'Awaiting Problem',
  'Awaiting User Info',
  'Awaiting Vendor',
  'Closed',
  'New',
  'Resolved',
  'Unknown'],
 'boolean_columns': ['active',
  'made_sla',
  'knowledge',
  'u_priority_confirmation']}

## 7. Duplicate Handling

The source is an event log (multiple rows per incident lifecycle).
We keep both:
- Clean event-level dataset (`incident_event_log_clean.csv`)
- Latest-state incident-level dataset (`incident_clean.csv`)

In [7]:
dup_incidents = df['number'].value_counts()
dup_count = int((dup_incidents > 1).sum())

df_event_clean = df.sort_values(['number', 'sys_updated_at']).reset_index(drop=True)
df_clean = df.sort_values('sys_updated_at', ascending=False)
df_clean = df_clean.drop_duplicates(subset=['number'], keep='first').reset_index(drop=True)

duplicate_summary = {
    'total_rows': int(len(df)),
    'unique_incidents': int(df['number'].nunique()),
    'incidents_with_multiple_rows': dup_count,
    'max_rows_per_incident': int(dup_incidents.max()),
    'mean_rows_per_incident': round(float(dup_incidents.mean()), 2),
    'event_level_rows': int(len(df_event_clean)),
    'incident_level_rows': int(len(df_clean)),
}

duplicate_summary

{'total_rows': 141712,
 'unique_incidents': 24918,
 'incidents_with_multiple_rows': 24918,
 'max_rows_per_incident': 58,
 'mean_rows_per_incident': 5.69,
 'event_level_rows': 141712,
 'incident_level_rows': 24918}

## 8. Data Validation

In [8]:
invalid_resolution_dates = (
    (df_clean['resolved_at'].notna()) &
    (df_clean['opened_at'].notna()) &
    (df_clean['resolved_at'] < df_clean['opened_at'])
).sum()

invalid_closed_dates = (
    (df_clean['closed_at'].notna()) &
    (df_clean['opened_at'].notna()) &
    (df_clean['closed_at'] < df_clean['opened_at'])
).sum()

bool_validity = {
    col: int((df_clean[col].isin([True, False])).sum())
    for col in BOOLEAN_COLUMNS
}

priority_checks = {
    'impact_level': sorted(df_clean['impact_level'].dropna().unique().tolist()),
    'urgency_level': sorted(df_clean['urgency_level'].dropna().unique().tolist()),
    'priority_level': sorted(df_clean['priority_level'].dropna().unique().tolist()),
}

validation_summary = {
    'resolved_before_opened': int(invalid_resolution_dates),
    'closed_before_opened': int(invalid_closed_dates),
    'boolean_valid_counts': bool_validity,
    'priority_levels_found': priority_checks,
}

validation_summary

{'resolved_before_opened': 0,
 'closed_before_opened': 0,
 'boolean_valid_counts': {'active': 24918,
  'made_sla': 24918,
  'knowledge': 24918,
  'u_priority_confirmation': 24918},
 'priority_levels_found': {'impact_level': [1, 2, 3],
  'urgency_level': [1, 2, 3],
  'priority_level': [1, 2, 3, 4]}}

## 9. Feature Engineering

In [9]:
df_clean['resolution_time_hours'] = (
    (df_clean['resolved_at'] - df_clean['opened_at']).dt.total_seconds() / 3600
).round(2)

df_clean['closure_time_hours'] = (
    (df_clean['closed_at'] - df_clean['opened_at']).dt.total_seconds() / 3600
).round(2)

df_clean['incident_year'] = df_clean['opened_at'].dt.year
df_clean['incident_month'] = df_clean['opened_at'].dt.month
df_clean['incident_day'] = df_clean['opened_at'].dt.day
df_clean['incident_dayofweek'] = df_clean['opened_at'].dt.dayofweek
df_clean['incident_quarter'] = df_clean['opened_at'].dt.quarter
df_clean['incident_week'] = df_clean['opened_at'].dt.isocalendar().week

day_names = {
    0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
    4: 'Friday', 5: 'Saturday', 6: 'Sunday',
}
df_clean['incident_day_name'] = df_clean['incident_dayofweek'].map(day_names)

df_clean['is_business_hours'] = (
    (df_clean['incident_dayofweek'] < 5) &
    (df_clean['opened_at'].dt.hour >= 9) &
    (df_clean['opened_at'].dt.hour < 18)
)

df_clean['is_high_priority'] = df_clean['priority_level'].isin([1, 2])
df_clean['is_high_impact'] = df_clean['impact_level'].isin([1, 2])
df_clean['is_high_urgency'] = df_clean['urgency_level'].isin([1, 2])

df_clean['sla_compliant'] = df_clean['made_sla'].astype(int)
df_clean['total_modifications'] = df_clean['sys_mod_count']
df_clean['total_rework'] = df_clean['reassignment_count'] + df_clean['reopen_count']

engineered_columns = [
    'resolution_time_hours', 'closure_time_hours', 'incident_year', 'incident_month',
    'incident_day', 'incident_dayofweek', 'incident_quarter', 'incident_week',
    'incident_day_name', 'is_business_hours', 'is_high_priority', 'is_high_impact',
    'is_high_urgency', 'sla_compliant', 'total_modifications', 'total_rework',
]

df_clean[engineered_columns].head()

,resolution_time_hours,closure_time_hours,incident_year,incident_month,incident_day,incident_dayofweek,incident_quarter,incident_week,incident_day_name,is_business_hours,is_high_priority,is_high_impact,is_high_urgency,sla_compliant,total_modifications,total_rework
0,119.73,240.03,2017,2,8,2,1,6,Wednesday,True,False,True,True,0,10,1
1,46.75,46.80,2017,2,15,2,1,7,Wednesday,False,False,True,True,1,2,0
2,2.35,2.35,2017,2,16,3,1,7,Thursday,True,False,True,True,1,3,1
3,0.73,0.73,2017,2,16,3,1,7,Thursday,True,False,True,True,1,3,1
4,NaN,21.88,2017,2,15,2,1,7,Wednesday,True,False,True,True,1,5,1


## 10. Final Quality Report

In [ ]:
remaining_nulls = df_clean.isnull().sum()
columns_with_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)

min_date = df_clean['opened_at'].min()
max_date = df_clean['opened_at'].max()

quality_summary = {
    'incident_level_rows': int(len(df_clean)),
    'incident_level_columns': int(len(df_clean.columns)),
    'memory_mb': float(round(df_clean.memory_usage(deep=True).sum() / 1024**2, 2)),
    'time_coverage_start': str(min_date.date()),
    'time_coverage_end': str(max_date.date()),
    'duration_days': int((max_date - min_date).days),
}

display(quality_summary)
if len(columns_with_nulls) > 0:
    display(columns_with_nulls)
else:
    display({'remaining_missing_columns': 0})

{'incident_level_rows': 24918,
 'incident_level_columns': 58,
 'memory_mb': np.float64(39.67),
 'time_coverage_start': '2016-02-29',
 'time_coverage_end': '2017-02-16',
 'duration_days': 353}

caused_by                24915
vendor                   24903
cmdb_ci                  24864
rfc                      24742
problem_id               24538
sys_created_at           11495
u_symptom                 5839
assignment_group          2157
resolution_time_hours     1556
resolved_at               1556
assigned_to                725
resolved_by                 99
subcategory                  8
category                     7
dtype: int64

## 11. Export Clean Datasets

In [11]:
final_checks = {
    'final_dataset_shape': df_clean.shape,
    'remaining_missing_values': int(df_clean.isna().sum().sum()),
    'duplicate_rows': int(df_clean.duplicated().sum()),
}

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
event_output_path = PROCESSED_DIR / 'incident_event_log_clean.csv'
incident_output_path = PROCESSED_DIR / 'incident_clean.csv'

df_event_clean.to_csv(event_output_path, index=False)
df_clean.to_csv(incident_output_path, index=False)

export_summary = {
    'event_output_path': str(event_output_path),
    'event_rows': int(len(df_event_clean)),
    'event_columns': int(len(df_event_clean.columns)),
    'incident_output_path': str(incident_output_path),
    'incident_rows': int(len(df_clean)),
    'incident_columns': int(len(df_clean.columns)),
}

display(final_checks)
display(export_summary)

{'final_dataset_shape': (24918, 58),
 'remaining_missing_values': 147404,
 'duplicate_rows': 0}

{'event_output_path': '..\\data\\processed\\incident_event_log_clean.csv',
 'event_rows': 141712,
 'event_columns': 42,
 'incident_output_path': '..\\data\\processed\\incident_clean.csv',
 'incident_rows': 24918,
 'incident_columns': 58}

## 12. Cleaning Decisions

| Issue | Action | Business Justification |
|---|---|---|
| Missing values | Kept or imputed depending on business meaning | Missing values can represent valid business information. |
| Date columns | Converted to datetime | Required for KPI calculations. |
| Duplicate rows | Validated before removal | Preserve event history whenever applicable. |
| Boolean values | Standardized | Ensure consistent analysis. |
| Derived features | Created | Support SLA and performance KPIs. |

## 13. Conclusion

The cleaning pipeline now delivers two reproducible outputs for downstream analysis:
- Event-level cleaned log for lifecycle/process analysis
- Incident-level latest-state table for KPI and dashboard analysis

What was completed in this notebook:
- Missing values handled with explicit, business-aware rules
- Date and categorical fields standardized
- Validation checks applied to key consistency rules
- Features engineered for analytical use cases
- Data exported for EDA, SQL analysis, and reporting

All operational interpretations and business insights are intentionally deferred to 03_Exploratory_Data_Analysis.ipynb.